# JUG 2026 Fraud detection (credit card transactions) 

## Supervised machine learning "binary classification" 
I will speak about **supervised machine learning binary classification**

in the context of fraud detection on the credit card transactions

# Prepartion Libraries, variables, data sets ...

In [ ]:
import sys
import os
from importlib import reload
fpath = os.path.join('..//scripts')
sys.path.append(fpath)

import warnings
warnings.filterwarnings('ignore')

#loading internal scripts
import datamanagement as dm
reload(dm)
import result as resultMd
reload(resultMd)
import graph as gf
reload(gf)

print('done')

In [ ]:
# standard librairies import
import numpy as np
import pandas as pd
import numpy
import pandas
import sklearn
import imblearn
import dataframe_image as dfi
from datetime import datetime

#diagrams
import seaborn as sns
import matplotlib.pyplot as plt

#Classifiers
from sklearn.dummy import DummyClassifier
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

#metrics
from sklearn.metrics import accuracy_score

#Scaling
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# word2vec
import gensim
from gensim.models import Word2Vec
from nltk.tokenize import sent_tokenize, word_tokenize

print('done')

In [ ]:
from IPython.display import display, Markdown

def IR(dfTrx, target='Class'):
    majorityCount=dfTrx[target].value_counts()[0]
    minorityCount=dfTrx[target].value_counts()[1]
    return majorityCount/minorityCount

def fraudRate(dfTrx, target='Class'):
    majorityCount=dfTrx[target].value_counts()[0]
    minorityCount=dfTrx[target].value_counts()[1]
    return minorityCount/(minorityCount+majorityCount)

def printMessage(message,figure,format="{0}"):
    fullMessage=message +" \033[1m\033[94m"+format+"\033[0m"
    print(fullMessage.format(figure))

# Data (Simulated data)

## Raw data

In [ ]:

folder='../data/raw/'
file='2018-05-01-raw.pkl'
with open(folder+file, 'rb') as file:
    dfTrxRaw = pd.read_pickle(file)
dfTrxRaw.head(7)

## Enriched data

In [ ]:
folder='../data/raw/'
file='2018-05-01.pkl'
with open(folder+file, 'rb') as file:
    dfEnriched = pd.read_pickle(file)
dfEnriched.head(7)

# IR Imbalance Ratio/ Fraud rate

## Data preparation

In [ ]:
row = 10000
cols = 5

Random = np.random.randint(low=0, high=100, size=(row, cols))

dfCreditApproval = pd.DataFrame(Random)

#print(df)

dfCreditApproval['Class'] = np.random.choice([0, 1], p=[0.75, 0.25], size=len(dfCreditApproval))
dfCreditApproval.Class.value_counts()

##########
row = 10000
cols = 5

Random = np.random.randint(low=0, high=100, size=(row, cols))

dfBankChurn = pd.DataFrame(Random)

#print(df)

dfBankChurn['Class'] = np.random.choice([0, 1], p=[0.8, 0.2], size=len(dfCreditApproval))
dfBankChurn.Class.value_counts()

#The value IR imbalance ratio is defined for the binary classification 
#by the ratio number of negative class examples on the number of positive class examples. 
#It should be greater or equal to 1.
#
#IR = (number positive examples) / (number of negative examples)
#IR >=1
#Because the fraud rate is very low we can set that
#IR =1/(Fraud rate at transaction level)


#-	Credit approval (refused 75%, approved 25%)
printMessage("IR for credit approval",IR(dfCreditApproval),"{0:.2f}")

#-	Bank churn (retained 80%, churn 20%) 
printMessage("IR for bank churn",IR(dfBankChurn),"{0:.2f}")

printMessage("IR for fraud detection",IR(dfEnriched,"TX_FRAUD"),"{0:.2f}")

## Count plot

In [ ]:
fig, ax =plt.subplots(1,2)
sns.countplot(x='Class', data=dfCreditApproval, ax=ax[0], orient="v") 
ax[0].set_title("Credit approval")
sns.countplot(x='TX_FRAUD', data=dfTrxRaw, ax=ax[1], orient="v") 
ax[1].set_title("Credit card fraud")
fig.show()

In [ ]:
# add pie diagram
fig, ax =plt.subplots(1,2)
fig.set_figheight(6)
fig.set_figwidth(12)
ax[0].pie(dfCreditApproval['Class'].value_counts(),autopct='%1.1f%%', labels=['Refused','Approved'], colors=['yellowgreen','r'])
ax[0].set_title("Credit approval")
ax[1].pie(dfTrxRaw['TX_FRAUD'].value_counts(),autopct='%1.1f%%', labels=['Genuine','Fraud'], colors=['yellowgreen','r'])
ax[1].set_title("Credit card fraud")
fig.show()

# Choose the good metrics

In [ ]:
dfLearning, dfValidation =dm.getDataLearningAndValidation()

dfLearning.head()

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split

TEST_SIZE = 0.20 # test size using_train_test_split
RANDOM_STATE = 0

predictors = dm.getPredictors(dfLearning)
target = dm.getTarget()

x_train, x_test, y_train, y_test = train_test_split(dfLearning[predictors], dfLearning[target], test_size = TEST_SIZE, 
                                                        stratify=dfLearning[target],
                                                        random_state = RANDOM_STATE)

modelClf = DummyClassifier(strategy="most_frequent")
modelClf.fit(x_train, y_train)

predsTrain = modelClf.predict(x_train)
predsTest = modelClf.predict(x_test)
predsValidation = modelClf.predict(dfValidation[predictors])

printMessage("","Dummy classifier")
printMessage("Accuracy score",accuracy_score(dfValidation[target],predsValidation),"{0:.4f}")

In [ ]:
from sklearn.metrics import f1_score

predsValidation = modelClf.predict(dfValidation[predictors])
printMessage("Accuracy =","(Num Correct Predictions) / ( Num Total Predictions)")
printMessage("Accuracy score",accuracy_score(dfValidation[target],predsValidation),"{0:.4f}")
printMessage("F1 score",f1_score(dfValidation[target],predsValidation),"{0:.4f}")
dm.show_confusion_matrix(dfValidation[target], predsValidation,'Confusion matrix validation data')

**F1 =2*True_Positive / (2 True_Positive + False_Negative + False_Positive)**

In [ ]:
from sklearn.naive_bayes import GaussianNB
modelClf = GaussianNB()
then= datetime.now()
modelClf.fit(x_train, y_train)
now = datetime.now()
duration= now - then
duration_in_s = duration.total_seconds()
print("Duration ",duration_in_s)

predsTrain = modelClf.predict(x_train)
predsTest = modelClf.predict(x_test)
predsValidation = modelClf.predict(dfValidation[predictors])
dm.show_confusion_matrix(dfValidation[target], predsValidation,'Confusion matrix validation data')
print((2*66)/(2*66+401+186))
printMessage("F1 score",f1_score(dfValidation[target],predsValidation),"{0:.4f}")

# Sampling

## PCA (Principal Component Analysis)

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from imblearn.under_sampling import RandomUnderSampler

predictors = dm.getPredictors(dfLearning)
target = dm.getTarget()

x1, y1 = dfLearning[predictors], dfLearning[target]
print("dfLearning ",dfLearning.shape)
print("x1         ",x1.shape)


pca = PCA(n_components=2)
sc = StandardScaler()
x2 = sc.fit_transform(x1)
x_pca = pca.fit_transform(x2)

# giving a larger plot
plt.figure(figsize=(8, 6))

plt.scatter(x_pca[:, 0], x_pca[:, 1],
            c=y1,
            cmap='plasma')

# labeling x and y axes
plt.xlabel('First Principal Component')
plt.ylabel('Second Principal Component')
plt.show()

## Under sampling (remove some geniune transactions) 

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from imblearn.under_sampling import RandomUnderSampler

predictors = dm.getPredictors(dfLearning)
target = dm.getTarget()

ratio=dfLearning[target].value_counts()[1]/dfLearning[target].value_counts()[0]
print(ratio)
undersample = RandomUnderSampler(sampling_strategy=1.2*ratio,random_state=42)
x1us, y1us = undersample.fit_resample(dfLearning[predictors], dfLearning[target])
print("dfLearning ",dfLearning.shape)
print("x1us         ",x1us.shape)

pca = PCA(n_components=2)
sc = StandardScaler()
x2 = sc.fit_transform(x1)
x_pca0 = pca.fit_transform(x2)


#------------------------
# giving a larger plot
fig,ax = plt.subplots(1,2)
fig.set_figheight(6)
fig.set_figwidth(12)
fig.subplots_adjust(hspace=0.5)
ax[0].set_title('Before sampling')
ax[0].scatter(x_pca0[:, 0], x_pca0[:, 1],
            c=y1,
            cmap='plasma')

sc = StandardScaler()
x2 = sc.fit_transform(x1us)
x_pca1 = pca.fit_transform(x2)
ax[1].set_title('After random under sampling')
ax[1].scatter(x_pca1[:, 0], x_pca1[:, 1],
            c=y1us,
            cmap='plasma')
plt.show()

In [ ]:
import pandas as pd 

def filterResult(inputData):
    output =inputData[(inputData.Name!='GaussianNB')]
    output =output[(output.Name!='DummyClassifier')]
    output =output[(output.Hyperparameters=='02-After tuning') | (output.Hyperparameters=='03-Random Undersampling')]
    return output

timeResponse = resultMd.load_time_response_result()
timeResponse= filterResult(timeResponse)


style_test=pd.DataFrame(data=timeResponse, 
             columns=['Package','Name','Hyperparameters','Learning time']).style.background_gradient(axis='index')
style_test.hide(axis="index")


style_test

## Over sampling (duplicate some fraudulent transactions) 

In [ ]:
from imblearn.over_sampling import RandomOverSampler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

predictors = dm.getPredictors(dfLearning)
target = dm.getTarget()

ratio=dfLearning[target].value_counts()[1]/dfLearning[target].value_counts()[0]
print(ratio)
oversample = RandomOverSampler(sampling_strategy=2*ratio,random_state=42)
x1os, y1os = oversample.fit_resample(dfLearning[predictors], dfLearning[target])
print("dfLearning ",dfLearning.shape)
print("x1us         ",x1us.shape)

pca = PCA(n_components=2)
sc = StandardScaler()
x2 = sc.fit_transform(x1)
x_pca0 = pca.fit_transform(x2)


#------------------------
# giving a larger plot
fig,ax = plt.subplots(1,2)
fig.set_figheight(6)
fig.set_figwidth(12)
fig.subplots_adjust(hspace=0.5)
ax[0].set_title('Before sampling')
ax[0].scatter(x_pca0[:, 0], x_pca0[:, 1],
            c=y1,
            cmap='plasma')

sc = StandardScaler()
x2 = sc.fit_transform(x1os)
x_pca1 = pca.fit_transform(x2)
ax[1].set_title('After random over sampling')
ax[1].scatter(x_pca1[:, 0], x_pca1[:, 1],
            c=y1os,
            cmap='plasma')
plt.show()

# Learning simple example with decision Tree

## split learning data / test data

In [ ]:
from sklearn.model_selection import train_test_split

TEST_SIZE = 0.20 # test size using_train_test_split
RANDOM_STATE = 0

predictors = dm.getPredictors(dfLearning)
target = dm.getTarget()

x_train, x_test, y_train, y_test = train_test_split(dfLearning[predictors], dfLearning[target], test_size = TEST_SIZE, 
                                                        stratify=dfLearning[target],
                                                        random_state = RANDOM_STATE)

print('train data',x_train.shape)
print('test data ',x_test.shape)
print(y_train.mean())
print(y_test.mean())

## Hyperparameters tuning search randomize 

**Cross validation**

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV

#{'criterion': 'entropy', 'max_depth': 9, 'min_samples_leaf': 7, 'min_samples_split': 32
dic_param={
    'criterion':["gini","entropy"],
    'max_depth': randint(2,12),
    'min_samples_leaf': randint(2,10),
    'min_samples_split': randint(10,20)
}
modelClf = DecisionTreeClassifier(random_state=42)

random_search = RandomizedSearchCV(modelClf,dic_param, scoring='f1', verbose=10,cv=4,n_iter=3).fit(x_train, y_train)
print(random_search.best_params_)
print(random_search.best_score_)

## Hyperparameters tuning grid search

**Exhautive search**

In [ ]:


from sklearn.tree import DecisionTreeClassifier
from scipy.stats import randint
from sklearn.model_selection import GridSearchCV

dic_param={
    'criterion':["gini","entropy"],
    'max_depth': [6,8],
    'min_samples_leaf': [2,3],
    'min_samples_split': [10,11]
}
modelClf = DecisionTreeClassifier(random_state=42)

random_search = GridSearchCV(modelClf,dic_param, scoring='f1', verbose=10,cv=4).fit(x_train, y_train)
print(random_search.best_params_)
print(random_search.best_score_)

## Results

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
from sklearn.tree import DecisionTreeClassifier
from datetime import datetime

modelClf = DecisionTreeClassifier(random_state=42)
parameters={'criterion': 'entropy', 'max_depth': 8, 'min_samples_leaf': 3, 'min_samples_split': 10}

modelClf.set_params(**parameters)
then= datetime.now()
modelClf.fit(x_train, y_train)
now = datetime.now()
duration= now - then
duration_in_s = duration.total_seconds()
print("Duration ",duration_in_s)


predsTrain = modelClf.predict(x_train)
predsTest = modelClf.predict(x_test)

F1Learning =f1_score(y_train, predsTrain)
F1Test=f1_score(y_test, predsTest)
dm.show_confusion_matrix(y_train, predsTrain,'Confusion matrix learning data')
print(f"f1 train {F1Learning:.3f}")
dm.show_confusion_matrix(y_test, predsTest,'Confusion matrix test data')
print(f"f1 test {F1Test:.3f}")


In [ ]:
gf.show_importance(modelClf, predictors)

In [ ]:
gf.show_prediction_graph(modelClf, x_test,y_test)

In [ ]:
predsValidation = modelClf.predict(dfValidation[predictors])
f1=f1_score(dfValidation[target], predsValidation)

dm.show_confusion_matrix(dfValidation[target], predsValidation,'Confusion matrix validation data')
print(f"f1 validation {f1:.3f}")